# Noise ablation on the model test split

Recomputes the morphometric indicators on the **test partition of the trained models only**, and
measures the same reconstructions **twice** — once with a synthetic noise realization, once without —
so the effect of `galmorph.data.add_noise` is isolated from everything else.

**Why.** The Fig. 4 comparison (`real_vs_reconstruction_morphology.ipynb`) showed that the
reconstructions carry about 11 % more background scatter than the real images once flux and size are
divided out, while the noise map itself matches the real background to 0.2 %. Two explanations remain,
and they call for opposite fixes:

1. the autoencoder passes input noise through, so `add_noise` adds a second sigma on top of a residual;
2. the decoder writes low-level structure (a halo, ringing) into the background, which inflates the
   measured scatter without being noise at all — that is reconstruction error, not a protocol bug.

Cell 6 separates them directly, by measuring the raw reconstruction before any noise is added. Leaked
pixel noise is white (lag-1 autocorrelation near 0); decoder structure is smooth (lag-1 near 1).

**Split convention.** Reproduces exactly what `Train-AE` uses everywhere
(`experiments/train_test.py`, `train_test_parallel.py`, `train_flow.py`, `verification.py`):

```python
load_dataset("VincentB03/euclid-Q1-VF", split="train").train_test_split(test_size=0.1, seed=42)["test"]
```

That is a *shuffled* split, so it cannot be reproduced with a `train[90%:]` slice — it has to be redone
with the same seed, which is what cell 5 does. It gives roughly 5 000 images out of 50 203, a sample
size comparable to the paper's, and keeps the measurement on data the model never saw.

The flow is deliberately left out: its noise handling needs a separate fix, and it is not needed for
these figures.

## 1. Colab setup

Both repos are public and neither is pip-installable: they are cloned and added to `sys.path`.
Skip cells 1-3 entirely if you are running locally with everything already installed.

In [ ]:
# --- Clone the repos (neither is pip-installable) ----------------------------
#   galaxy-morphometrics -> galmorph  (measurements)
#   Train-AE             -> pshear    (model definition, to load the checkpoint)
import os, sys, subprocess

REPOS = {
    "galaxy-morphometrics": "https://github.com/VincentB03/galaxy-morphometrics.git",
    "Train-AE":             "https://github.com/VincentB03/Train-AE.git",
}

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

for name, url in REPOS.items():
    if not os.path.isdir(name) and IN_COLAB:
        subprocess.run(["git", "clone", "--depth", "1", url, name], check=True)
    path = os.path.abspath(name)
    if os.path.isdir(path):
        if path not in sys.path:
            sys.path.insert(0, path)
        print(f"{name:22s} on sys.path -> {path}")
    else:
        # running locally: assume it is already importable
        print(f"{name:22s} not cloned here; assuming it is already importable")


In [ ]:
# --- Python dependencies -----------------------------------------------------
# Train-AE's requirements pin equinox, needed to read the .eqx checkpoint.
# galsim (galmorph) and jax-galsim (pshear) are different packages: both are needed.
# jax is left to the runtime so that its CUDA build matches (Colab ships one).
if IN_COLAB:
    !pip install -q -r Train-AE/requirements.txt
    !pip install -q -r galaxy-morphometrics/requirements.txt

# Fail now rather than halfway through the reconstruction.
import importlib.util

REQUIRED = ("galsim", "jax_galsim", "jax", "equinox", "flowjax", "einops",
            "astropy", "datasets", "rpy2", "wandb", "yaml")
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError(
        "missing modules: " + ", ".join(missing)
        + "\nIf `galsim` is the one missing, see the GalSim install instructions "
          "linked from galaxy-morphometrics/README.md -- the pip build fails on "
          "some platforms."
    )

import pshear, galmorph
print("pshear   ->", pshear.__file__)
print("galmorph ->", galmorph.__file__)


In [ ]:
# --- R + SDMTools for CAS / Gini-M20 / MID (README procedure, ~3-4 min) ------
if IN_COLAB:
    !apt-get -qq install -y r-base > /dev/null
    !R -e 'install.packages("R.utils", repos="https://cloud.r-project.org")' > /dev/null 2>&1
    !wget -q https://cran.r-project.org/src/contrib/Archive/SDMTools/SDMTools_1.1-221.2.tar.gz
    !tar xzf SDMTools_1.1-221.2.tar.gz
    !sed -i '9a #include <math.h>\n#define PI M_PI' SDMTools/src/pointinpolygon.c
    !sed -i '7a #define PI M_PI' SDMTools/src/vincenty.geodesics.c
    !R CMD INSTALL SDMTools > /dev/null 2>&1

# Check the R setup before loading the data.
import rpy2.robjects as ro
ro.r('library(SDMTools)')
print("R + SDMTools OK")

## 2. Configuration

In [ ]:
# --- Everything you may need to edit -----------------------------------------
DATASET      = "VincentB03/euclid-Q1-VF"
IMAGE_FIELD  = "sci_subtracted"
PSF_FIELD    = "psf_residual"    # not "psf_stamp": the AE was trained on psf_residual
NOISE_FIELD  = "noise_map"
MASK_FIELD   = "binary_mask"     # dataset convention: 1 = valid, 0 = defective

# Split: must match Train-AE exactly, or the "test" set is not the models' test set.
TEST_SIZE    = 0.1
SPLIT_SEED   = 42
SPLIT_NAME   = "test"

# Autoencoder: the run train_flow.py points at (download_wandb_weights.py names an
# older one, i1pf186a). With a pre-populated wandb_weights/ cache, the WandB API is
# skipped and WANDB_PROJECT does not matter.
WANDB_ENTITY  = "vincentb03-imt-atlantique"
WANDB_PROJECT = "Test-AE-partial-3"   # <-- CHECK THIS for run i344nq38
AE_RUN_ID     = "i344nq38"
AE_EPOCH      = 2000
CACHE_DIR     = "wandb_weights"
AE_RUN_PATH   = f"{WANDB_ENTITY}/{WANDB_PROJECT}/{AE_RUN_ID}"

# Measurement
PIXEL_SCALE  = 0.1      # arcsec/pixel, matches the AE config (scale: 0.1)
STAMP_SIZE   = 64       # matches the AE config (nx: 64, ny: 64)
MORPH_CROP   = None     # stamps are already 64x64, nothing to crop
NOISE_SEED   = 0
POOL_SIZE    = 4        # worker processes for the statistics
N_MAX        = None     # e.g. 800 for a quick pass; None = the whole test split
BORDER       = 6        # width of the background strip used for the direct measurements

# Apply the real mask to the reconstructions too, so both sets reject the same stamps
# (without it, the full run rejected 2.82 % of real vs 0.31 % of reconstructions).
APPLY_MASK_TO_RECON = True

In [ ]:
from getpass import getpass
import os

if IN_COLAB and not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")

HF_TOKEN = os.environ.get("HF_TOKEN")

## 3. Load the models' test split

In [ ]:
import numpy as np
from datasets import load_dataset

ds_full = load_dataset(DATASET, split="train", token=HF_TOKEN)
ds = ds_full.train_test_split(test_size=TEST_SIZE, seed=SPLIT_SEED)[SPLIT_NAME]
print(f"full dataset {len(ds_full)} -> {SPLIT_NAME} split {len(ds)}")

if N_MAX is not None:
    ds = ds.select(range(min(N_MAX, len(ds))))
    print(f"subsampled to {len(ds)}")

ds = ds.with_format("numpy")
images = np.asarray(ds[IMAGE_FIELD],  dtype=np.float64)
psf    = np.asarray(ds[PSF_FIELD],    dtype=np.float64)
nmap   = np.asarray(ds[NOISE_FIELD],  dtype=np.float64)

# Flip to galmorph's convention (nonzero = bad), as load_hf_stamps does.
bad = (np.asarray(ds[MASK_FIELD], dtype=np.float64) == 0).astype(np.float64)

for name, a in [("images", images), ("psf", psf), ("noise_map", nmap), ("bad", bad)]:
    print(f"  {name:10s} {a.shape}  [{np.nanmin(a):+.4g}, {np.nanmax(a):+.4g}]")
assert images.shape[-1] == STAMP_SIZE, f"stamps are {images.shape[-1]}px, STAMP_SIZE says {STAMP_SIZE}"
print(f"\nmasked pixels: {100*bad.mean():.3f} % of all pixels, "
      f"{100*(bad.reshape(len(bad), -1).mean(1) > 0).mean():.2f} % of stamps affected")

## 4. Reconstruct once

The reconstruction is computed **once** and reused for both variants, so the only difference between
them is the noise. Running the CLI twice would re-run the autoencoder and also redraw the latent
sample, leaving two variables changed instead of one.

In [ ]:
from galmorph.autoencoder import WandBGalaxyAutoencoder

ae = WandBGalaxyAutoencoder(AE_RUN_PATH, AE_EPOCH, cache_dir=CACHE_DIR)
recon_clean = ae.reconstruct(images, psf=psf)
print("reconstruction:", recon_clean.shape)

## 5. The decisive measurement

Before computing any indicator: measure the background scatter of the **raw** reconstruction, with no
noise added. The lag-1 autocorrelation is what separates the two hypotheses.

- residual scatter ≈ 0 → the reconstruction really is noise-free, and the 11 % excess came from
  somewhere else;
- residual scatter ≈ 0.4–0.5 σ **and lag-1 ≈ 0** → leaked pixel noise, `add_noise` is double-counting:
  inject in quadrature;
- residual scatter ≈ 0.4–0.5 σ **and lag-1 large** → not noise but smooth decoder structure in the
  background. Injecting less noise would be papering over a reconstruction defect.

In [ ]:
def nmad(a, axis=None):
    med = np.median(a, axis=axis, keepdims=True)
    return 1.4826 * np.median(np.abs(a - med), axis=axis)

edge = np.zeros((STAMP_SIZE, STAMP_SIZE), dtype=bool)
edge[:BORDER, :] = edge[-BORDER:, :] = True
edge[:, :BORDER] = edge[:, -BORDER:] = True

def lag1(stack):
    """Horizontal lag-1 autocorrelation over the top border strip."""
    strip = stack[:, :BORDER, :]
    return np.corrcoef(strip[:, :, :-1].ravel(), strip[:, :, 1:].ravel())[0, 1]

sig_real  = np.median(nmad(images[:, edge],      axis=1))
sig_clean = np.median(nmad(recon_clean[:, edge], axis=1))
sig_map   = np.median(np.median(nmap[:, edge],   axis=1))

print(f"background scatter, real images        sigma_real  = {sig_real:.6g}")
print(f"noise map over the same pixels         sigma_map   = {sig_map:.6g}   "
      f"(ratio {sig_real/sig_map:.3f})")
print(f"background scatter, RAW reconstruction sigma_clean = {sig_clean:.6g}   "
      f"({sig_clean/sig_real:.3f} x sigma_real)")
print()
print(f"lag-1 autocorrelation, real images        {lag1(images):+.3f}")
print(f"lag-1 autocorrelation, RAW reconstruction {lag1(recon_clean):+.3f}")
print()

f = sig_clean / sig_real
predicted = np.sqrt(f**2 + 1.0)
print(f"if a full sigma is added on top, expected total = sqrt({f:.3f}^2 + 1) = {predicted:.3f} x sigma_real")
print(f"quadrature-corrected injection would be sigma_add = sqrt(1 - {f:.3f}^2) = "
      f"{np.sqrt(max(1 - f**2, 0)):.3f} x sigma")

# --- Background scatter in annuli --------------------------------------------
# In the catalogues, the excess is largest for small, faint galaxies (sigma ratio
# ~1.27) and vanishes for large or bright ones (~0.96): structure close to the source?
yy, xx = np.mgrid[0:STAMP_SIZE, 0:STAMP_SIZE]
rr = np.hypot(xx - (STAMP_SIZE - 1) / 2.0, yy - (STAMP_SIZE - 1) / 2.0)

print(f"{'annulus (px)':>14s} {'sigma_real':>13s} {'sigma_clean':>13s} {'ratio':>8s}")
for r0, r1 in [(8, 12), (12, 16), (16, 20), (20, 24), (24, 32)]:
    ann = (rr >= r0) & (rr < r1)
    if ann.sum() < 20:
        continue
    a = np.median(nmad(images[:, ann], axis=1))
    b = np.median(nmad(recon_clean[:, ann], axis=1))
    print(f"{f'[{r0}, {r1})':>14s} {a:13.6g} {b:13.6g} {b/a:8.3f}")

print()
print("ratio flat with radius     -> a uniform residual, consistent with leaked noise")
print("ratio falling with radius  -> decoder structure concentrated near the source,")
print("                              i.e. reconstruction error, not a noise-injection bug")


## 6. Measure the indicators three ways

In [ ]:
from galmorph.data import add_noise
from galmorph.pipeline import compute_statistics

recon_noisy = add_noise(recon_clean, nmap, seed=NOISE_SEED)

datasets = {
    "real":         images,
    "reco_nonoise": recon_clean,
    "reco_noise":   recon_noisy,
}
masks = {"real": bad}
if APPLY_MASK_TO_RECON:
    masks["reco_nonoise"] = bad
    masks["reco_noise"]   = bad

print(f"measuring {len(datasets)} x {len(images)} stamps, this is the slow part...")
tables = compute_statistics(
    datasets,
    pixel_scale=PIXEL_SCALE,
    morph_crop=MORPH_CROP,
    pool_size=POOL_SIZE,
    compute_morph=True,
    masks=masks,
)
for name, t in tables.items():
    print(f"  {name:13s} {len(t)} rows")

In [ ]:
# --- Summary ------------------------------------------------------------------
COLS = ["sn", "Gini", "C", "A", "M20", "size"]

def dist_median(t, col):
    x = np.asarray(t[col], float)
    m = np.asarray(t["flag_morph"]) & np.isfinite(x) & (x != -9)
    return (np.median(x[m]) if m.any() else np.nan), int(m.sum())

print("selection: how many stamps survive the R measurement")
for name, t in tables.items():
    fm = np.asarray(t["flag_morph"])
    print(f"  {name:13s} {fm.sum():6d} / {len(t):6d} kept   ({100*(1-fm.mean()):5.2f} % rejected)")

ref = tables["real"]
print(f"\n{'col':6s} {'real':>10s} {'reco_nonoise':>14s} {'reco_noise':>12s}"
      f" {'nonoise-real':>14s} {'noise-real':>12s}")
for col in COLS:
    if col not in ref.colnames:
        continue
    mr, _ = dist_median(ref, col)
    mc, _ = dist_median(tables["reco_nonoise"], col)
    mn, _ = dist_median(tables["reco_noise"], col)
    print(f"{col:6s} {mr:10.4f} {mc:14.4f} {mn:12.4f}"
          f" {100*(mc/mr-1):13.1f}% {100*(mn/mr-1):11.1f}%")

In [ ]:
# --- Noise amplitude implied by each variant ----------------------------------
# sn ~ SB_central / sigma_noise  and  SB_central ~ flux / size^2, so dividing out the
# HSM flux (`amp`) and size (`sigma_e`) isolates the noise term.
def moment_ratio(t, col):
    x = np.asarray(ref[col], float); y = np.asarray(t[col], float)
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if "flag_moments" in ref.colnames:
        m &= np.asarray(ref["flag_moments"]) & np.asarray(t["flag_moments"])
    return np.median(y[m] / x[m])

sn_real, _ = dist_median(ref, "sn")
print(f"{'variant':14s} {'R_sn':>7s} {'R_amp':>7s} {'R_size':>7s} {'sigma_reco/sigma_real':>22s}")
for name in ("reco_nonoise", "reco_noise"):
    t = tables[name]
    sn_v, _ = dist_median(t, "sn")
    R_sn   = sn_real / sn_v
    R_amp  = moment_ratio(t, "amp")
    R_size = moment_ratio(t, "sigma_e")
    print(f"{name:14s} {R_sn:7.4f} {R_amp:7.4f} {R_size:7.4f} {R_sn*R_amp/R_size**2:22.4f}")

print("\nreco_noise should land near 1.00 if add_noise is doing the right thing.")
print("reco_nonoise is a control: with no noise added its sigma ratio is the residual")
print("scatter of the raw reconstruction, the same quantity cell 5 measured on the pixels.")

## 7. Fig. 4 panels, with and without the injected noise

In [ ]:
import matplotlib.pyplot as plt

PANELS = [("M20", r"$M_{20}$"), ("Gini", r"Gini $G$"), ("C", r"Concentration $C$"), ("A", r"Asymmetry $A$")]

def paired(t, col):
    x = np.asarray(ref[col], float); y = np.asarray(t[col], float)
    m = (np.asarray(ref["flag_morph"]) & np.asarray(t["flag_morph"])
         & np.isfinite(x) & np.isfinite(y) & (x != -9) & (y != -9))
    return x[m], y[m]

fig, axes = plt.subplots(2, len(PANELS), figsize=(4 * len(PANELS), 8.4))
for row, variant in enumerate(("reco_noise", "reco_nonoise")):
    for ax, (col, label) in zip(axes[row], PANELS):
        x, y = paired(tables[variant], col)
        lo, hi = np.percentile(np.concatenate([x, y]), [0.5, 99.5])
        pad = 0.05 * (hi - lo); lim = (lo - pad, hi + pad)
        box = (x >= lim[0]) & (x <= lim[1]) & (y >= lim[0]) & (y <= lim[1])
        ax.hexbin(x[box], y[box], gridsize=45, extent=(*lim, *lim), cmap="Blues", mincnt=1, linewidths=0)
        ax.plot(lim, lim, ls=":", color="k", lw=1.2)
        ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
        bias = np.median(y - x)
        nmad_ = 1.4826 * np.median(np.abs((y - x) - bias))
        ax.set_title(f"{label}\n{variant}: bias {bias:+.4f}, NMAD {nmad_:.4f}", fontsize=10)
        ax.set_xlabel("original"); ax.set_ylabel("reconstruction")
plt.tight_layout(); plt.show()

## 8. How to read this

Compare the two rows of panels, and the two rows of the sigma table.

**If `reco_noise` gives sigma ≈ 1.00 and `reco_nonoise` gives sigma well below 1**, the noise
injection is working and the noise-free variant is simply under-noised — the expected result, and
panel A should be visibly better behaved in the top row.

**If `reco_noise` still comes out above 1.05**, `add_noise` is over-noising, and cell 5 says which
fix applies. Watch the asymmetry `A` in particular: it is the indicator that subtracts a background
term estimated from the noise (`A = (Agal - (ngal/nbkg)*Abkg)/(2*Aden)` in `compute_CA.R`), so it
moves first and moves most. In the full-catalogue run it was off by −13.5 % while Gini moved only
−0.5 %.

**The `reco_nonoise` column is not a candidate protocol.** Without noise, `segmap` thresholds on
quantiles of an image whose background is nearly constant, and the S/N estimator divides by a
background scatter that tends to zero. Its indicators are not comparable to the real ones — it is a
control that isolates the residual, not an alternative way of measuring.

**One caveat on the quadrature fix.** Matching total variance is not matching the noise. The real
background is spatially correlated (lag-1 ≈ +0.22, from Lanczos3 resampling), `add_noise` injects
white noise, and any residual from the decoder is smooth. Three different power spectra with the same
sigma still give different `segmap`, Gini and `A`. Equalising the amplitude is the first step, not
the last one.